# Séance 5 — Analyse exploratoire orientée décision
*Session 5 — Decision-oriented exploratory analysis*

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/fraude-360-cours/blob/main/cours/laboratoires/05_exploration.ipynb)

**Cours :** Science des données appliquée : détection de la fraude transactionnelle (45 h) · **Chargé de cours :** Noé Carème Fouotsa Manfouo · **Durée :** 3 h

| Parcours de formation | Cette séance |
|---|---|
| Gestion — UdeS École de gestion (SIMQG), Bishop's Williams School | ● séance complète |
| Informatique — UdeS maîtrise/bac en informatique (IA et science des données) | ● séance complète |
| Cégep / formation continue — Cégep de Sherbrooke (AEC), UdeS formation continue | ● séance complète |

**À la fin de cette séance, vous saurez :**

1. mesurer le taux de fraude et le **lift** par modalité d'une variable ;
2. repérer les profils temporels (heure, jour) de la fraude ;
3. classer les variables selon leur pouvoir discriminant univarié (AUC) ;
4. transformer une exploration en messages compréhensibles par un gestionnaire.

**Mode d'emploi :** exécutez les cellules dans l'ordre (`Maj + Entrée`). Complétez les cellules `# VOTRE CODE ICI`, puis
lancez la cellule ✅ qui suit pour obtenir une rétroaction immédiate. Chaque exercice indique le ou les parcours concernés :
🟢 tous · 🟣 Gestion + Informatique · 🔷 Informatique · 🔶 Gestion. Les exercices 🔷/🔶 sont facultatifs pour les autres parcours.

In [ ]:
# --- Configuration (une fois par session) ----------------------------------------
import sys, pathlib, subprocess, shutil
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not pathlib.Path("fraude-360-cours").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/fraude-360-cours.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "fraude-360-cours/cours/requirements-colab.txt"], check=True)
    if shutil.which("java") is None:   # PySpark a besoin d'un Java 11, 17 ou 21
        subprocess.run("apt-get -qq install -y openjdk-17-jre-headless > /dev/null", shell=True, check=True)
    RACINE = pathlib.Path("fraude-360-cours").resolve()
else:
    ici = pathlib.Path.cwd().resolve()
    RACINE = ici.parents[1] if ici.name == "laboratoires" else ici
sys.path.insert(0, str(RACINE / "cours"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from outils_cours import preparer, config, table, verifier, verifier_proche, image, lire_json
pd.set_option("display.max_columns", 40)
print("Racine du projet :", RACINE)

In [ ]:
cfg = preparer("eda")
synthese = lire_json(cfg.chemin("artefacts") / "eda" / "synthese_eda.json")
print({k: v for k, v in synthese.items() if not isinstance(v, (list, dict))})
for f in ["01_serie_temporelle.png", "02_typologies.png", "06_pouvoir_discriminant.png"]:
    image(cfg.chemin("artefacts") / "eda" / f)

In [ ]:
gold = table("gold", "caracteristiques")
taux_global = gold["est_fraude"].mean()
print(f"{len(gold):,} transactions, taux de fraude {taux_global:.3%}")

### Exercice 5.1 — 🟢 Tous les parcours

Écrivez `taux_par_modalite(df, col)` qui renvoie un `DataFrame` indexé par les modalités de `col`, avec les colonnes
`nb` (effectif), `taux_fraude` et `lift` (= taux de la modalité / taux global), trié par `lift` décroissant. Appliquez-la à `canal`.

In [ ]:
# VOTRE CODE ICI
def taux_par_modalite(df, col):
    ...

taux_par_modalite(gold, "canal")
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
_t = taux_par_modalite(gold, "canal")
verifier(list(_t.columns[:3]) == ["nb", "taux_fraude", "lift"] and _t["lift"].is_monotonic_decreasing)
verifier_proche((_t.nb * _t.lift).sum() / _t.nb.sum(), 1.0, 1e-9, "La moyenne pondérée des lifts vaut 1")

### Exercice 5.2 — 🟢 Tous les parcours

Calculez `taux_heure`, le taux de fraude par heure (`heure`, 0 à 23), et tracez-le en barres. À quelles heures le
risque est-il le plus élevé ?

In [ ]:
# VOTRE CODE ICI
taux_heure = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier(len(taux_heure) == 24 and taux_heure.index.min() == 0)
verifier(taux_heure.loc[0:5].mean() > taux_heure.loc[9:17].mean(), "La fraude est plus fréquente la nuit")

### Exercice 5.3 — 🟣 Gestion + Informatique

Pour chaque variable **numérique** de la spécification, calculez l'AUC univariée `max(AUC, 1 − AUC)` avec
`sklearn.metrics.roc_auc_score` (valeurs manquantes remplacées par la médiane). Stockez dans `auc_univariee` une `Series` triée
par ordre décroissant et affichez les 10 premières.

In [ ]:
# VOTRE CODE ICI
from sklearn.metrics import roc_auc_score
spec = lire_json(config().chemin("artefacts") / "variables" / "specification_courante.json")
auc_univariee = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier(auc_univariee.is_monotonic_decreasing and auc_univariee.between(0.5, 1).all())
verifier(auc_univariee.iloc[0] > 0.7, "Les meilleures variables séparent nettement fraude et légitime")

**❓ Question (🔶 Gestion — décision d'affaires) —** Choisissez trois constats de cette exploration et rédigez-les en une phrase chacun pour un directeur de la prévention de la fraude (sans jargon statistique).

*Votre réponse ici.*

## Bilan

- Le lift traduit un taux en « fois plus risqué que la moyenne » : parlant pour le métier.
- L'AUC univariée classe rapidement les variables, mais ignore les interactions (rôle des modèles).
- Une bonne exploration produit des messages actionnables, pas seulement des graphiques.

**Pour aller plus loin :**

- `artifacts/eda/rapport_eda.html` : le rapport complet généré par le pipeline.

---
*Fraude-360 — données entièrement synthétiques. Cours conçu par Noé Carème Fouotsa Manfouo · licence MIT (code) et CC BY 4.0 (contenu pédagogique).*